# 01 · Contingency analysis (3A): N-1, storm-corridor N-k and load at risk

**Project:** Grid stress test — contingency analysis, heat derating and cascading impacts on critical services (Nagpur, Maharashtra)

**Question for this notebook:** which transmission elements are critical, and how much load is at risk when one element fails (N-1) or when a windstorm takes out several at once (N-k)?

| Step | Method | Main reference |
|---|---|---|
| Base case | AC power flow (Newton-Raphson) in pandapower | Thurner et al. (2018) |
| N-1 | each line and transformer out in turn: overloads, voltage violations, non-convergence, islanding | Stott, Alsac & Monticelli (1987); Wood & Wollenberg (2013) |
| Ranking | load shed, overloads and the Ejebe–Wollenberg performance index | Ejebe & Wollenberg (1979) |
| Load at risk | DC optimal power flow with sheddable loads (linear programme) | Salmeron et al. (2004) |
| N-k | storm corridor + tower fragility curves + Monte Carlo | Panteli et al. (2017) |
| Restoration | crew-limited greedy repair → resilience trapezoid | Panteli & Mancarella (2015) |

**Run order:** this is the first notebook. Its outputs are used by notebook 03 (heat rerun) and notebook 04 (cascades).

**Outputs** (in `results/01_contingency/`): `network_prepared.json`, `n1_results.csv`, `critical_lines.csv`, `storm_sweep.csv`, `storm_mc_summary.csv`, `load_at_risk.csv`, `scenarios_meta.csv`, `bus_shed_timeseries.csv`, `network_buses.csv`, figures.

**Data:** IEEE 118-bus test case (built into pandapower). No external downloads are needed in this notebook.

## 0. Setup

In [ ]:
!pip install -q pandapower

In [ ]:
import os, sys, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 3)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})

# ---- Shared project settings (identical in all five notebooks) ----
PROJECT_ID = "goggle-earth-499404"          # Google Cloud project registered for Earth Engine
SITE = "Nagpur, Maharashtra"
LAT, LON, ELEV_M = 21.15, 79.09, 310
TZ = "Asia/Kolkata"

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True                             # keep results in Google Drive so the notebooks can share them
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/grid_stress_nagpur"
else:
    ROOT = os.path.abspath("grid_stress_nagpur")

CACHE = os.path.join(ROOT, "cache")
RES = {k: os.path.join(ROOT, "results", k) for k in
       ["01_contingency", "02_feeder", "03_heat", "04_cascade", "05_mitigation"]}
for d in [CACHE, *RES.values()]:
    os.makedirs(d, exist_ok=True)
OUT = RES["01_contingency"]

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(OUT, name), dpi=150, bbox_inches="tight")
    plt.show()

def require(path, notebook):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run notebook {notebook} first (with USE_DRIVE = True).")
    return path

print("Project folder:", ROOT)

## 1. Study parameters
All assumptions are collected here so they can be quoted in the technical note.

- **Ratings.** pandapower's `case118` ships without thermal limits (9,900 MVA everywhere). We rate every branch so that its base-case flow uses at most `TARGET_LOADING` of its rating, with a minimum rating per voltage level. This is a documented synthetic choice, common for IEEE test cases.
- **VOLL** (value of lost load) only needs to be much larger than generation costs, so the optimiser sheds load only as a last resort.
- **Re-dispatch.** `REDISPATCH_FRAC = None` allows full corrective re-dispatch (optimistic). Set it to e.g. 0.2 so each generator can only increase output by 20% of its capacity within the emergency window (more realistic).
- **Storm.** Nagpur's IS 875 (Part 3) basic wind speed is 44 m/s (50-year, 3-s gust). The two storm scenarios exceed this design value, as damaging events do. Fragility parameters are illustrative and should be stated as assumptions.

In [ ]:
CASE = "case118"                 # "case118" (recommended for N-k) or "case30"
RATING_MODE = "base_flow"        # "base_flow" (assign ratings) or "case" (keep ratings shipped with the case)
TARGET_LOADING = 0.70            # base-case flow / rating
FLOOR_MVA = {138: 100, 161: 150, 345: 400, 132: 100, 33: 30}   # minimum rating by voltage level (kV: MVA)

VOLL = 10000.0                   # value of lost load (cost units per MWh), >> generation cost
REDISPATCH_FRAC = None           # None = unlimited corrective re-dispatch; 0.2 = +20% of capacity

REPAIR_H_N1 = 8                  # outage duration for a single-element (N-1) failure, hours

# Storm-corridor hazard
BOX_KM = 250                     # synthetic geographic extent of the test network around Nagpur
STORMS = {"severe": 50.0, "extreme": 55.0}   # peak 3-s gust on the storm track, m/s
SIGMA_KM = 12.0                  # width parameter of the gust footprint (Gaussian), km
TRACK_KM = 400                   # length of the storm track, km
SWEEP_BEARINGS = [0, 45, 90, 135]
SWEEP_OFFSETS = [-80, -40, 0, 40, 80]
FRAGILITY_MEDIAN = 65.0          # tower fragility: gust at which a tower has 50% failure probability, m/s
FRAGILITY_BETA = 0.12            # lognormal dispersion
SPAN_KM = 0.35                   # tower spacing, km
N_MC_SWEEP = 30                  # Monte Carlo samples per corridor in the sweep
N_MC = 500                       # Monte Carlo samples on the design corridor
N_MC_RESTORE = 30                # samples carried through restoration and exported to notebook 04

# Restoration (Cyclone Fani 2019 showed transmission restoration over days)
N_CREWS = 3
REPAIR_H_STORM = 24.0            # hours per crew per failed line (tower replacement / restringing)
DELAY_H = 6.0                    # damage assessment and mobilisation before repairs start

SEED = 42
rng = np.random.default_rng(SEED)

## 2. Core functions
These functions are reused unchanged in notebook 03 for the heat-derated rerun.

**AC contingency screening.** Set the element(s) out of service, run a full AC power flow, record:
- branch loadings above 100% of rating (thermal violations),
- bus voltages outside the case limits (0.94–1.06 p.u. for case118),
- non-convergence (a sign of voltage collapse),
- buses isolated from the slack (islanding), and the load on them,
- the performance index \(PI = \sum_l \frac{1}{2n}\left(\frac{S_l}{S_l^{max}}\right)^{2n}\) with \(n=1\) (Ejebe & Wollenberg, 1979).

**Load at risk — DC optimal power flow with sheddable loads.** For each outage we solve the linear programme

$$\min_{\theta, P_g, s}\; \text{VOLL}\sum_i s_i + \sum_g c_g P_g$$
$$\text{s.t.}\quad B\,\theta = C_g P_g - (P_d - s), \qquad |b_l(\theta_{f(l)} - \theta_{t(l)})| \le S^{max}_l,$$
$$0 \le P_g \le P_g^{max}, \qquad 0 \le s_i \le P_{d,i}, \qquad \theta_{ref,k} = 0 \text{ in every island } k.$$

The DC approximation (flat voltages, lossless lines, small angle differences) keeps the problem linear and always solvable, and handles islands naturally: an island with no generator must shed all its load. The minimum load shed \(\sum s_i\) is the **load at risk**: the load that cannot be served even after the best possible re-dispatch. AC screening tells us *whether* limits are violated; the LP tells us *how much load* is lost to fix them.

In [ ]:
# ---- Core grid functions (shared by notebooks 01 and 03) ----
import copy
import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import pandapower.topology as top
import scipy.sparse as sp
from scipy.optimize import linprog

SQRT3 = np.sqrt(3.0)


def branch_flows_mva(net):
    """Apparent power through each line and transformer (max of both ends), MVA."""
    rl = net.res_line
    s_line = np.maximum(np.hypot(rl.p_from_mw, rl.q_from_mvar), np.hypot(rl.p_to_mw, rl.q_to_mvar))
    if len(net.trafo):
        rt = net.res_trafo
        s_tr = np.maximum(np.hypot(rt.p_hv_mw, rt.q_hv_mvar), np.hypot(rt.p_lv_mw, rt.q_lv_mvar))
    else:
        s_tr = pd.Series(dtype=float)
    return s_line, s_tr


def assign_ratings(net, mode="base_flow", target_loading=0.7, floor_mva=None):
    """Give the test network realistic thermal ratings.

    mode="case": keep the ratings that come with the test case.
    mode="base_flow": rating = max(|S_base| / target_loading, voltage-level floor).
      This makes the base case secure (every branch <= target_loading) while leaving
      limited headroom, a common way to rate IEEE cases that ship without limits.
    Transformer impedances are kept constant when their MVA rating is changed.
    """
    if mode == "case":
        return net
    floor_mva = floor_mva or {}
    pp.runpp(net)
    s_line, s_tr = branch_flows_mva(net)
    vn_line = net.bus.loc[net.line.from_bus, "vn_kv"].values
    floor_l = np.array([floor_mva.get(v, 0.0) for v in vn_line])
    rating_l = np.maximum(s_line.values / target_loading, floor_l)
    rating_l = np.ceil(rating_l / 5) * 5                                 # round up to 5 MVA steps
    net.line["max_i_ka"] = rating_l / (SQRT3 * vn_line * net.line.parallel.values)
    if len(net.trafo):
        vn_tr = net.bus.loc[net.trafo.hv_bus, "vn_kv"].values
        floor_t = np.array([floor_mva.get(v, 0.0) for v in vn_tr])
        rating_t = np.ceil(np.maximum(s_tr.values / target_loading, floor_t) / 5) * 5
        k = rating_t / net.trafo.sn_mva.values                           # keep ohmic impedance unchanged
        net.trafo["vk_percent"] *= k
        net.trafo["vkr_percent"] *= k
        net.trafo["sn_mva"] = rating_t
    pp.runpp(net)
    return net


def branch_table(net):
    """All lines and transformers with DC-model reactance (p.u. on net.sn_mva) and MVA rating."""
    base = net.sn_mva
    L = net.line
    vn = net.bus.loc[L.from_bus, "vn_kv"].values
    zb = vn ** 2 / base
    lines = pd.DataFrame({
        "kind": "line", "idx": L.index.values,
        "f": L.from_bus.values, "t": L.to_bus.values,
        "x_pu": L.x_ohm_per_km.values * L.length_km.values / L.parallel.values / zb,
        "rating_mva": SQRT3 * vn * L.max_i_ka.values * L.parallel.values * L.df.values,
        "in_service": L.in_service.values,
    })
    tabs = [lines]
    if len(net.trafo):
        T = net.trafo
        vk = np.sqrt(np.maximum(T.vk_percent.values ** 2 - T.vkr_percent.values ** 2, 1e-9)) / 100
        tabs.append(pd.DataFrame({
            "kind": "trafo", "idx": T.index.values,
            "f": T.hv_bus.values, "t": T.lv_bus.values,
            "x_pu": vk * base / T.sn_mva.values / T.parallel.values,
            "rating_mva": T.sn_mva.values * T.parallel.values,
            "in_service": T.in_service.values,
        }))
    bt = pd.concat(tabs, ignore_index=True)
    bt["name"] = bt["kind"].str[0].str.upper() + bt["idx"].astype(str) + " (" + bt["f"].astype(str) + "-" + bt["t"].astype(str) + ")"
    return bt


def bus_loads(net):
    """Net active demand per bus (loads minus static generators), MW."""
    d = net.load[net.load.in_service].groupby("bus")["p_mw"].sum() if len(net.load) else pd.Series(dtype=float)
    if len(net.sgen):
        d = d.sub(net.sgen[net.sgen.in_service].groupby("bus")["p_mw"].sum(), fill_value=0)
    return d.reindex(net.bus.index, fill_value=0.0)


def generator_table(net):
    g = net.gen[net.gen.in_service]
    e = net.ext_grid[net.ext_grid.in_service]
    cost = {}
    if len(net.poly_cost):
        for _, r in net.poly_cost.iterrows():
            cost[(r.et, r.element)] = r.cp1_eur_per_mw
    gens = pd.DataFrame({
        "et": ["gen"] * len(g) + ["ext_grid"] * len(e),
        "element": list(g.index) + list(e.index),
        "bus": list(g.bus) + list(e.bus),
        "p0": list(g.p_mw) + [np.nan] * len(e),
        "pmax": list(g.max_p_mw) + list(e.max_p_mw if "max_p_mw" in e else [1e4] * len(e)),
    })
    gens["pmax"] = gens["pmax"].fillna(1e4)
    gens["cost"] = [cost.get((a, b), 20.0) for a, b in zip(gens.et, gens.element)]
    return gens


class DCLoadShed:
    """DC optimal power flow that treats every load as sheddable.

    minimise   VOLL * sum(shed) + sum(c_g * P_g)
    subject to B*theta = Cg*P_g - (P_d - shed)      (nodal balance, per unit)
               |b_l (theta_f - theta_t)| <= rating_l   (thermal limits)
               0 <= P_g <= P_g,max ; 0 <= shed <= P_d
               theta = 0 at one bus in every island
    Islands with no generation are forced to shed all their load.
    """

    def __init__(self, net, voll=10000.0, redispatch_frac=None, rating_scale=None):
        self.net = net
        self.base = net.sn_mva
        self.bt = branch_table(net)
        if rating_scale is not None:                      # e.g. heat derating of lines only
            self.bt.loc[self.bt.kind == "line", "rating_mva"] *= rating_scale
        self.buses = net.bus.index[net.bus.in_service].values
        self.pos = {b: i for i, b in enumerate(self.buses)}
        self.pd = bus_loads(net).reindex(self.buses).clip(lower=0).values
        self.gens = generator_table(net)
        self.voll = voll
        self.redispatch_frac = redispatch_frac
        self.key = {(k, i): n for n, (k, i) in enumerate(zip(self.bt.kind, self.bt.idx))}

    def solve(self, outages=(), return_flows=False):
        bt = self.bt
        on = bt.in_service.values.copy()
        for o in outages:
            on[self.key[o]] = False
        br = bt[on]
        n, m, ng = len(self.buses), len(br), len(self.gens)
        f = np.array([self.pos[b] for b in br.f]); t = np.array([self.pos[b] for b in br.t])
        b = 1.0 / br.x_pu.values
        C = sp.csr_matrix((np.r_[np.ones(m), -np.ones(m)], (np.r_[np.arange(m), np.arange(m)], np.r_[f, t])), shape=(m, n))
        Bf = sp.diags(b) @ C                                   # branch flow = Bf @ theta  (p.u.)
        Bbus = C.T @ Bf
        gb = np.array([self.pos[x] for x in self.gens.bus])
        Cg = sp.csr_matrix((np.ones(ng), (gb, np.arange(ng))), shape=(n, ng))
        # islands -> one reference angle each
        import networkx as nx
        G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(zip(f, t))
        islands = list(nx.connected_components(G))
        refs = [min(c) for c in islands]
        # variables: theta (n) | Pg (ng) | shed (n)
        nv = n + ng + n
        A_eq = sp.hstack([Bbus, -Cg, -sp.eye(n)]).tocsr()
        b_eq = -self.pd / self.base
        R = sp.csr_matrix((np.ones(len(refs)), (np.arange(len(refs)), refs)), shape=(len(refs), nv))
        A_eq = sp.vstack([A_eq, R]).tocsr(); b_eq = np.r_[b_eq, np.zeros(len(refs))]
        rate = br.rating_mva.values / self.base
        A_ub = sp.vstack([sp.hstack([Bf, sp.csr_matrix((m, ng + n))]),
                          sp.hstack([-Bf, sp.csr_matrix((m, ng + n))])]).tocsr()
        b_ub = np.r_[rate, rate]
        pmax = self.gens.pmax.values.copy()
        if self.redispatch_frac is not None:                  # limited corrective re-dispatch
            p0 = self.gens.p0.values
            up = np.where(np.isnan(p0), pmax, np.minimum(pmax, p0 + self.redispatch_frac * pmax))
            pmax = up
        bounds = [(None, None)] * n + [(0, p / self.base) for p in pmax] + [(0, d / self.base) for d in self.pd]
        cost = np.r_[np.zeros(n), self.gens.cost.values / 1000.0, np.full(n, self.voll / 1000.0)]
        res = linprog(cost, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
        if res.status != 0:
            return dict(ok=False, shed_mw=np.nan, shed_bus=None, n_islands=len(islands))
        x = res.x
        shed = x[n + ng:] * self.base
        out = dict(ok=True, shed_mw=float(shed.sum()), shed_bus=pd.Series(shed, index=self.buses),
                   n_islands=len(islands), pg=x[n:n + ng] * self.base)
        if return_flows:
            out["flows"] = pd.Series(Bf @ x[:n] * self.base, index=br.index)
        return out


def set_status(net, outages, status):
    for kind, idx in outages:
        net[kind].at[idx, "in_service"] = status


def ac_contingency(net, outages, pi_n=1):
    """AC power flow with the given (kind, idx) elements out of service. Restores the network afterwards."""
    set_status(net, outages, False)
    r = dict(converged=False)
    try:
        unsup = top.unsupplied_buses(net)
        r["n_isolated_buses"] = len(unsup)
        r["ac_load_isolated_mw"] = float(net.load[net.load.bus.isin(unsup) & net.load.in_service].p_mw.sum())
        try:
            pp.runpp(net, init="dc")
        except pp.LoadflowNotConverged:
            pp.runpp(net, init="flat", max_iteration=50)
        r["converged"] = True
        ll = net.res_line.loading_percent.where(net.line.in_service).dropna()
        tl = net.res_trafo.loading_percent.where(net.trafo.in_service).dropna() if len(net.trafo) else pd.Series(dtype=float)
        load = pd.concat([ll, tl])
        r["max_loading_pct"] = float(load.max())
        r["n_overloads"] = int((load > 100).sum())
        r["overload_excess_pct"] = float((load - 100).clip(lower=0).sum())
        r["overloaded"] = ";".join([f"L{i}" for i in ll[ll > 100].index] + [f"T{i}" for i in tl[tl > 100].index])
        r["PI"] = float(((load / 100) ** (2 * pi_n)).sum() / (2 * pi_n))       # Ejebe & Wollenberg (1979)
        vm = net.res_bus.vm_pu.dropna()
        vmin_lim = net.bus.loc[vm.index, "min_vm_pu"].fillna(0.95)
        vmax_lim = net.bus.loc[vm.index, "max_vm_pu"].fillna(1.05)
        r["vmin_pu"], r["vmax_pu"] = float(vm.min()), float(vm.max())
        r["n_voltage_violations"] = int(((vm < vmin_lim) | (vm > vmax_lim)).sum())
    except pp.LoadflowNotConverged:
        pass
    finally:
        set_status(net, outages, True)
    return r


def run_contingency_set(net, outage_list, voll=10000.0, redispatch_frac=None, rating_scale=None, label=""):
    """AC screening + DC load-shed for a list of outage sets. Returns a DataFrame."""
    dc = DCLoadShed(net, voll=voll, redispatch_frac=redispatch_frac, rating_scale=rating_scale)
    rows = []
    for outages in outage_list:
        r = ac_contingency(net, outages)
        s = dc.solve(outages)
        r.update(outage=";".join(f"{k[0].upper()}{i}" for k, i in outages), k=len(outages),
                 dc_ok=s["ok"], load_shed_mw=s["shed_mw"], n_islands=s["n_islands"])
        r["violation"] = (not r["converged"]) or r.get("n_overloads", 0) > 0 or r.get("n_voltage_violations", 0) > 0
        rows.append(r)
    df = pd.DataFrame(rows)
    df.insert(0, "case", label)
    return df

## 3. Prepare the network and run the base case

In [ ]:
net = getattr(pn, CASE)()
assign_ratings(net, RATING_MODE, TARGET_LOADING, FLOOR_MVA)
pp.runpp(net)

bt = branch_table(net)
s_line, s_tr = branch_flows_mva(net)
base = pd.DataFrame({
    "buses": len(net.bus), "lines": len(net.line), "transformers": len(net.trafo),
    "generators": len(net.gen) + len(net.ext_grid), "loads": len(net.load),
    "total load (MW)": net.load.p_mw.sum(),
    "generation capacity (MW)": generator_table(net).pmax.sum(),
    "max branch loading (%)": max(net.res_line.loading_percent.max(), net.res_trafo.loading_percent.max() if len(net.trafo) else 0),
    "V min / max (p.u.)": f"{net.res_bus.vm_pu.min():.3f} / {net.res_bus.vm_pu.max():.3f}",
}, index=[CASE]).T
pp.to_json(net, os.path.join(OUT, "network_prepared.json"))
base

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(pd.concat([net.res_line.loading_percent, net.res_trafo.loading_percent]), bins=30, color="#2c6fbb")
ax[0].set_xlabel("Base-case loading (% of rating)"); ax[0].set_ylabel("Branches")
ax[0].set_title("Branch loading, base case")
ax[1].hist(net.res_bus.vm_pu, bins=30, color="#4a9c5d")
ax[1].set_xlabel("Voltage (p.u.)"); ax[1].set_title("Bus voltages, base case")
savefig("fig01_base_case.png")

### 3.1 Validate the DC model against pandapower
With generator outputs fixed to pandapower's DC power-flow solution and limits removed, our LP must reproduce pandapower's DC branch flows. Small differences come from transformer tap ratios, which our simplified model ignores.

In [ ]:
pp.rundcpp(net)
chk = DCLoadShed(net, voll=1e6)
chk.bt["rating_mva"] = 1e6
pg_dc = np.r_[net.res_gen.p_mw.values, net.res_ext_grid.p_mw.values]
chk.gens = chk.gens.assign(pmax=pg_dc, cost=-1.0)          # forces every generator to its DC set-point
v = chk.solve(return_flows=True)
pp_flow = np.r_[net.res_line.p_from_mw.values, net.res_trafo.p_hv_mw.values if len(net.trafo) else []]
err = np.abs(v["flows"].values - pp_flow)
print(f"Load shed in validation run: {v['shed_mw']:.3f} MW (should be 0)")
print(f"Max |flow difference|: {err.max():.2f} MW; mean: {err.mean():.2f} MW; largest flow: {np.abs(pp_flow).max():.0f} MW")
pp.runpp(net)    # restore AC results

## 4. N-1 contingency analysis
Every line and transformer is removed in turn (186 cases for case118).

In [ ]:
t0 = time.time()
n1_list = [[("line", i)] for i in net.line.index] + [[("trafo", i)] for i in net.trafo.index]
n1 = run_contingency_set(net, n1_list, voll=VOLL, redispatch_frac=REDISPATCH_FRAC, label="baseline")
n1 = n1.merge(bt[["name", "kind", "idx", "rating_mva"]].assign(outage=lambda d: d.kind.str[0].str.upper() + d.idx.astype(str)),
              on="outage", how="left")
print(f"{len(n1)} contingencies in {time.time() - t0:.0f}s")

summary_n1 = pd.Series({
    "contingencies": len(n1),
    "with any violation": int(n1.violation.sum()),
    "  thermal overloads": int((n1.n_overloads > 0).sum()),
    "  voltage violations": int((n1.n_voltage_violations > 0).sum()),
    "  non-converged": int((~n1.converged).sum()),
    "causing islanding": int((n1.n_isolated_buses > 0).sum()),
    "with load at risk > 0 (DC-OPF)": int((n1.load_shed_mw > 0.1).sum()),
    "max load at risk (MW)": n1.load_shed_mw.max(),
})
summary_n1.to_frame("N-1, baseline ratings")

### 4.1 Ranked list of critical elements
Ranking order: load at risk (DC-OPF) first, then number of overloads, total overload above 100%, and the performance index. An element is critical if losing it forces load shedding or pushes other branches past their limits.

In [ ]:
n1 = n1.sort_values(["load_shed_mw", "n_overloads", "overload_excess_pct", "PI"],
                    ascending=False, na_position="first").reset_index(drop=True)
n1["rank"] = n1.index + 1
n1.to_csv(os.path.join(OUT, "n1_results.csv"), index=False)
cols = ["rank", "name", "rating_mva", "load_shed_mw", "ac_load_isolated_mw", "n_overloads", "max_loading_pct",
        "overloaded", "n_voltage_violations", "vmin_pu", "PI"]
critical = n1[cols].head(20)
critical.to_csv(os.path.join(OUT, "critical_lines.csv"), index=False)
critical

In [ ]:
top = n1.head(15).iloc[::-1]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
ax[0].barh(top["name"], top["load_shed_mw"], color="#c0392b"); ax[0].set_xlabel("Load at risk (MW, DC-OPF)")
ax[1].barh(top["name"], top["max_loading_pct"], color="#2c6fbb"); ax[1].axvline(100, color="k", lw=0.8)
ax[1].set_xlabel("Highest post-contingency loading (%)")
fig.suptitle(f"{CASE}: 15 most critical single-element outages", fontsize=10)
savefig("fig02_critical_elements.png")

**Reading the results.** Many N-1 overloads disappear in the DC-OPF because re-dispatching generators relieves them. These are *operational* violations, not load loss. Contingencies with load at risk usually isolate part of the network (a radial bus or a transformer feeding a sub-area). AC isolated load can exceed the DC value because the AC model drops islands without a slack bus, while the LP lets local generators in an island keep serving load.

## 5. Hazard footprint: storm-corridor N-k
### 5.1 Hazard model
1. **Placement.** The test network has no real geography. We place pandapower's generic bus coordinates in a 250 km × 250 km square centred on Nagpur, so a storm has a spatial footprint. This placement is synthetic.
2. **Wind field.** A straight storm track (squall line / downburst swath) with peak gust \(V_{peak}\) on the track, decaying with distance \(d\):
\(V(d) = V_{peak}\,\exp\!\left[-\tfrac12 (d/\sigma)^2\right]\).
3. **Fragility.** Each tower fails with probability \(p_{tower}(V) = \Phi\!\left(\ln(V/V_{50})/\beta\right)\) (lognormal fragility curve). A line is a chain of towers spaced `SPAN_KM` apart and fails if any tower fails:
\(P_{line} = 1 - \prod_j \left(1 - p_{tower}(V_j)\right)\) (Panteli et al., 2017).
4. **Sampling.** Monte Carlo draws which lines fail; each draw is an N-k contingency evaluated with the DC-OPF.
5. **Restoration.** After a mobilisation delay, `N_CREWS` crews each repair one line every `REPAIR_H_STORM` hours, choosing the repair that restores most load first.

In [ ]:
# ---- Storm-corridor hazard model (notebook 01) ----
import json
import numpy as np
import pandas as pd
from scipy.stats import norm


def bus_coordinates(net, box_km=250.0, centre=(21.15, 79.09)):
    """Place the (non-geographic) test network in a box_km x box_km square centred on the site.
    Uses pandapower's generic bus coordinates; the placement is synthetic and only gives the
    hazard footprint a spatial meaning."""
    xy = np.array([json.loads(g)["coordinates"] for g in net.bus.geo])
    xy = (xy - xy.min(0)) / (xy.max(0) - xy.min(0)).max()              # keep aspect ratio
    xy = (xy - xy.max(0) / 2) * box_km                                  # km, centred on 0,0
    lat0, lon0 = centre
    df = pd.DataFrame({"x_km": xy[:, 0], "y_km": xy[:, 1]}, index=net.bus.index)
    df["lat"] = lat0 + df.y_km / 110.574
    df["lon"] = lon0 + df.x_km / (111.320 * np.cos(np.radians(lat0)))
    return df


def line_geometry(net, coords):
    f = coords.loc[net.line.from_bus].values[:, :2]
    t = coords.loc[net.line.to_bus].values[:, :2]
    g = pd.DataFrame({"x0": f[:, 0], "y0": f[:, 1], "x1": t[:, 0], "y1": t[:, 1]}, index=net.line.index)
    g["length_km"] = np.hypot(g.x1 - g.x0, g.y1 - g.y0).clip(lower=1.0)
    return g


def dist_to_segment(px, py, ax, ay, bx, by):
    abx, aby = bx - ax, by - ay
    tt = np.clip(((px - ax) * abx + (py - ay) * aby) / max(abx ** 2 + aby ** 2, 1e-12), 0, 1)
    return np.hypot(px - (ax + tt * abx), py - (ay + tt * aby))


def corridor_endpoints(offset_km, bearing_deg, length_km):
    """Straight storm track crossing the box. bearing = direction of travel (0 = north, 90 = east);
    offset = perpendicular shift of the track from the box centre."""
    th = np.radians(bearing_deg)
    d = np.array([np.sin(th), np.cos(th)])          # unit vector of travel (x = east, y = north)
    nrm = np.array([d[1], -d[0]])
    c = nrm * offset_km
    return c - d * length_km / 2, c + d * length_km / 2


def gust_field(px, py, track, v_peak, sigma_km, v_background=0.0):
    """Peak gust (m/s) at points: Gaussian decay with distance from the storm track."""
    (ax, ay), (bx, by) = track
    d = dist_to_segment(px, py, ax, ay, bx, by)
    return v_background + (v_peak - v_background) * np.exp(-0.5 * (d / sigma_km) ** 2)


def tower_fragility(v, median=65.0, beta=0.12):
    """Probability that one tower/span fails at gust v (lognormal fragility curve)."""
    v = np.maximum(np.asarray(v, float), 1e-6)
    return norm.cdf(np.log(v / median) / beta)


def line_failure_prob(geom, track, v_peak, sigma_km, span_km=0.35, median=65.0, beta=0.12):
    """Each line is a chain of towers; it fails if any tower fails: P = 1 - prod(1 - p_tower)."""
    p, vmax = [], []
    for _, g in geom.iterrows():
        n = max(int(np.ceil(g.length_km / span_km)), 1)
        s = (np.arange(n) + 0.5) / n
        x = g.x0 + s * (g.x1 - g.x0); y = g.y0 + s * (g.y1 - g.y0)
        v = gust_field(x, y, track, v_peak, sigma_km)
        pt = tower_fragility(v, median, beta)
        p.append(1 - np.exp(np.sum(np.log1p(-np.minimum(pt, 1 - 1e-12)))))
        vmax.append(v.max())
    return pd.DataFrame({"p_fail": p, "v_max": vmax}, index=geom.index)


def restoration_curve(dc, failed, n_crews=3, repair_h=24.0, delay_h=6.0, horizon_h=None):
    """Greedy restoration: after an assessment delay, each crew repairs one line every repair_h hours;
    the next line repaired is the one that reduces load shed the most.
    Returns hourly shed per bus (DataFrame hours x buses) and the repair order."""
    remaining = list(failed)
    events = []                                   # (time_h, list of outages still active)
    t = delay_h
    current = dc.solve(remaining)
    timeline = [(0.0, current["shed_bus"])]
    order = []
    while remaining:
        batch = []
        for _ in range(min(n_crews, len(remaining))):
            best, best_shed, best_res = None, np.inf, None
            for o in remaining:
                trial = [x for x in remaining if x != o]
                r = dc.solve(trial)
                s = r["shed_mw"] if r["ok"] else np.inf
                if s < best_shed - 1e-6:
                    best, best_shed, best_res = o, s, r
            if best is None:                          # no single repair helps: take the first
                best = remaining[0]; best_res = dc.solve(remaining[1:])
            remaining.remove(best); batch.append(best)
            current = best_res
        t += repair_h
        order.append((t, batch))
        timeline.append((t, current["shed_bus"]))
    end = int(np.ceil(timeline[-1][0])) if horizon_h is None else horizon_h
    hours = np.arange(end + 1)
    rows = []
    for h in hours:
        sb = [s for (tt, s) in timeline if tt <= h][-1]
        rows.append(sb.values)
    shed = pd.DataFrame(rows, index=hours, columns=timeline[0][1].index)
    return shed, order

In [ ]:
coords = bus_coordinates(net, BOX_KM, (LAT, LON))
geom = line_geometry(net, coords)
dc = DCLoadShed(net, voll=VOLL, redispatch_frac=REDISPATCH_FRAC)
print(f"Synthetic line lengths: median {geom.length_km.median():.0f} km, range {geom.length_km.min():.0f}–{geom.length_km.max():.0f} km")

vv = np.linspace(20, 80, 200)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(vv, tower_fragility(vv, FRAGILITY_MEDIAN, FRAGILITY_BETA), color="#c0392b")
ax[0].axvline(44, color="grey", ls="--", lw=1); ax[0].text(44.5, 0.6, "IS 875 basic\nwind speed,\nNagpur (44 m/s)", fontsize=7)
ax[0].set_xlabel("3-s gust (m/s)"); ax[0].set_ylabel("P(tower failure)"); ax[0].set_title("Tower fragility curve")
n_tow = int(geom.length_km.median() / SPAN_KM)
ax[1].plot(vv, 1 - (1 - tower_fragility(vv, FRAGILITY_MEDIAN, FRAGILITY_BETA)) ** n_tow, color="#2c6fbb")
ax[1].set_xlabel("3-s gust along whole line (m/s)"); ax[1].set_ylabel("P(line failure)")
ax[1].set_title(f"Line of median length ({n_tow} towers)")
savefig("fig03_fragility.png")

### 5.2 Corridor sweep — which storm track hurts most?
We test every combination of track direction and offset, with a small Monte Carlo per corridor, and take the track with the largest expected load at risk as the **design corridor** for each storm intensity.

In [ ]:
def mc_corridor(pf, n, rng):
    shed, nfail = [], []
    for _ in range(n):
        failed = [("line", i) for i in pf.index[rng.random(len(pf)) < pf.p_fail.values]]
        r = dc.solve(failed)
        shed.append(r["shed_mw"]); nfail.append(len(failed))
    return np.array(shed), np.array(nfail)

t0 = time.time()
sweep = []
for storm, vpk in STORMS.items():
    for b in SWEEP_BEARINGS:
        for off in SWEEP_OFFSETS:
            tr = corridor_endpoints(off, b, TRACK_KM)
            pf = line_failure_prob(geom, tr, vpk, SIGMA_KM, SPAN_KM, FRAGILITY_MEDIAN, FRAGILITY_BETA)
            shed, nf = mc_corridor(pf, N_MC_SWEEP, rng)
            sweep.append(dict(storm=storm, v_peak=vpk, bearing=b, offset_km=off,
                              expected_failures=pf.p_fail.sum(), mean_shed_mw=shed.mean(), p95_shed_mw=np.percentile(shed, 95)))
sweep = pd.DataFrame(sweep)
sweep.to_csv(os.path.join(OUT, "storm_sweep.csv"), index=False)
design = sweep.loc[sweep.groupby("storm")["mean_shed_mw"].idxmax()].set_index("storm")
print(f"Sweep done in {time.time() - t0:.0f}s")
design

In [ ]:
def draw_network(ax, line_values=None, cmap="Reds", vmax=1.0):
    from matplotlib.collections import LineCollection
    segs = [[(g.x0, g.y0), (g.x1, g.y1)] for _, g in geom.iterrows()]
    if line_values is None:
        lc = LineCollection(segs, colors="#8a8a8a", linewidths=0.8)
    else:
        lc = LineCollection(segs, array=np.asarray(line_values), cmap=cmap, linewidths=1.6, norm=plt.Normalize(0, vmax))
    ax.add_collection(lc)
    load = bus_loads(net)
    ax.scatter(coords.x_km, coords.y_km, s=4 + load.values / 8, color="#333333", zorder=3)
    gb = net.gen.bus.tolist() + net.ext_grid.bus.tolist()
    ax.scatter(coords.loc[gb, "x_km"], coords.loc[gb, "y_km"], marker="s", s=14, color="#2c6fbb", zorder=4)
    ax.set_aspect("equal"); ax.set_xlabel("km east of Nagpur (synthetic)"); ax.set_ylabel("km north")
    return lc

fig, axes = plt.subplots(1, len(STORMS), figsize=(5.5 * len(STORMS), 5.6), layout="constrained")
PF_DESIGN = {}
for ax, (storm, row) in zip(np.atleast_1d(axes), design.iterrows()):
    tr = corridor_endpoints(row.offset_km, row.bearing, TRACK_KM)
    pf = line_failure_prob(geom, tr, row.v_peak, SIGMA_KM, SPAN_KM, FRAGILITY_MEDIAN, FRAGILITY_BETA)
    PF_DESIGN[storm] = (tr, pf)
    gx, gy = np.meshgrid(np.linspace(-BOX_KM / 2, BOX_KM / 2, 150), np.linspace(-BOX_KM / 2, BOX_KM / 2, 150))
    cs = ax.contourf(gx, gy, gust_field(gx, gy, tr, row.v_peak, SIGMA_KM), levels=[30, 40, 44, 50, 55, 60], cmap="Blues", alpha=0.35)
    lc = draw_network(ax, pf.p_fail.values)
    ax.set_xlim(-BOX_KM / 2, BOX_KM / 2); ax.set_ylim(-BOX_KM / 2, BOX_KM / 2)
    ax.set_title(f"{storm} storm ({row.v_peak:.0f} m/s), bearing {row.bearing:.0f}°, offset {row.offset_km:.0f} km", fontsize=9)
fig.colorbar(lc, ax=axes, shrink=0.7, label="P(line failure)", location="right")
fig.colorbar(cs, ax=axes, shrink=0.5, label="peak gust (m/s)", location="bottom")
plt.savefig(os.path.join(OUT, "fig04_design_corridors.png"), dpi=150, bbox_inches="tight"); plt.show()

### 5.3 Monte Carlo on the design corridors

In [ ]:
mc_rows, line_freq = [], {}
MC_SAMPLES = {}
for storm, (tr, pf) in PF_DESIGN.items():
    shed, nfail, samples = [], [], []
    for k in range(N_MC):
        failed = [("line", i) for i in pf.index[rng.random(len(pf)) < pf.p_fail.values]]
        r = dc.solve(failed)
        shed.append(r["shed_mw"]); nfail.append(len(failed)); samples.append(failed)
    shed, nfail = np.array(shed), np.array(nfail)
    MC_SAMPLES[storm] = (samples, shed)
    fails = pd.Series([i for s in samples for (_, i) in s]).value_counts() / N_MC
    line_freq[storm] = fails
    mc_rows.append(dict(storm=storm, v_peak=STORMS[storm], samples=N_MC,
                        mean_lines_failed=nfail.mean(), p95_lines_failed=np.percentile(nfail, 95),
                        p_any_shed=(shed > 0.1).mean(), mean_shed_mw=shed.mean(),
                        p50_shed_mw=np.median(shed), p95_shed_mw=np.percentile(shed, 95), max_shed_mw=shed.max(),
                        mean_shed_pct_of_load=100 * shed.mean() / net.load.p_mw.sum()))
mc_summary = pd.DataFrame(mc_rows).set_index("storm")
mc_summary.to_csv(os.path.join(OUT, "storm_mc_summary.csv"))
mc_summary.T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for storm, (samples, shed) in MC_SAMPLES.items():
    s = np.sort(shed)
    ax[0].plot(s, 1 - np.arange(len(s)) / len(s), label=f"{storm} ({STORMS[storm]:.0f} m/s)")
ax[0].set_xlabel("Load at risk (MW)"); ax[0].set_ylabel("P(load at risk ≥ x)"); ax[0].legend(frameon=False)
ax[0].set_title("Exceedance curve of load at risk")
worst = list(MC_SAMPLES)[-1]
lf = line_freq[worst].head(12).iloc[::-1]
ax[1].barh([bt.set_index(["kind", "idx"]).loc[("line", i), "name"] for i in lf.index], lf.values, color="#c0392b")
ax[1].set_xlabel("Failure frequency in Monte Carlo"); ax[1].set_title(f"Most exposed lines, {worst} storm")
savefig("fig05_storm_mc.png")

### 5.4 Restoration and outage duration
For each storm, we follow (a) the **expected footprint** (lines with \(P_{line} \ge 0.5\)) and (b) `N_MC_RESTORE` Monte Carlo samples through crew-limited restoration. This gives hourly load shed at every bus. The resulting curve is the degradation–restoration part of the resilience trapezoid (Panteli & Mancarella, 2015), and the per-bus outage durations feed notebook 04.

In [ ]:
REST = {}          # scenario_id -> (hourly shed per bus DataFrame, failed list, family)
t0 = time.time()
for storm, (tr, pf) in PF_DESIGN.items():
    expected = [("line", i) for i in pf.index[pf.p_fail >= 0.5]]
    if expected:                                   # a weaker storm may have no line with P >= 0.5
        shed_t, order = restoration_curve(dc, expected, N_CREWS, REPAIR_H_STORM, DELAY_H)
        REST[f"storm_{storm}_expected"] = (shed_t, expected, f"storm_{storm}_expected")
    samples, shed = MC_SAMPLES[storm]
    idx = [i for i in np.argsort(-shed) if shed[i] > 0.1]
    pick = idx[:: max(1, len(idx) // N_MC_RESTORE)][:N_MC_RESTORE]     # spread over the distribution
    for j, k in enumerate(pick):
        shed_t, _ = restoration_curve(dc, samples[k], N_CREWS, REPAIR_H_STORM, DELAY_H)
        REST[f"storm_{storm}_mc{j:03d}"] = (shed_t, samples[k], f"storm_{storm}_mc")
print(f"{len(REST)} restoration runs in {time.time() - t0:.0f}s")

fig, ax = plt.subplots(figsize=(8, 3.6))
colors = {"severe": "#e08e0b", "extreme": "#c0392b"}
for sid, (shed_t, failed, fam) in REST.items():
    storm = sid.split("_")[1]
    tot = shed_t.sum(axis=1)
    if fam.endswith("mc"):
        ax.step(tot.index, tot.values, where="post", color=colors[storm], alpha=0.15, lw=0.8)
    else:
        ax.step(tot.index, tot.values, where="post", color=colors[storm], lw=2.2, label=f"{storm}: expected footprint ({len(failed)} lines)")
ax.set_xlim(0, 120); ax.set_xlabel("Hours after storm"); ax.set_ylabel("Load not served (MW)")
ax.set_title("Load at risk through restoration (thin lines: Monte Carlo samples)"); ax.legend(frameon=False)
savefig("fig06_restoration.png")

## 6. Load-at-risk table and export for notebook 04

In [ ]:
total_load = net.load.p_mw.sum()
rows = []
for _, r in n1[n1.load_shed_mw > 0.1].iterrows():
    rows.append(dict(scenario_id=f"n1_{r.outage}", family="N-1", outages=r.outage, n_out=1,
                     peak_shed_mw=r.load_shed_mw, duration_h=REPAIR_H_N1, unserved_mwh=r.load_shed_mw * REPAIR_H_N1))
for sid, (shed_t, failed, fam) in REST.items():
    tot = shed_t.sum(axis=1)
    if tot.max() <= 0.1:
        continue
    rows.append(dict(scenario_id=sid, family=fam, outages=";".join(f"L{i}" for _, i in failed), n_out=len(failed),
                     peak_shed_mw=tot.max(), duration_h=int((tot > 0.1).sum()), unserved_mwh=tot.sum()))
meta = pd.DataFrame(rows)
meta["peak_shed_pct"] = 100 * meta.peak_shed_mw / total_load
meta.to_csv(os.path.join(OUT, "scenarios_meta.csv"), index=False)

lar = meta.groupby("family").agg(scenarios=("scenario_id", "size"), elements_out=("n_out", "mean"),
                                 peak_shed_mw_mean=("peak_shed_mw", "mean"), peak_shed_mw_max=("peak_shed_mw", "max"),
                                 duration_h_mean=("duration_h", "mean"), unserved_mwh_mean=("unserved_mwh", "mean"))
lar.to_csv(os.path.join(OUT, "load_at_risk.csv"))
lar

In [ ]:
# Hourly shed per bus for every scenario (only buses that lose load), used by notebook 04
load_bus = bus_loads(net)
ts = []
for _, r in n1[n1.load_shed_mw > 0.1].iterrows():
    kind = "line" if r.outage[0] == "L" else "trafo"
    s = dc.solve([(kind, int(r.outage[1:]))])["shed_bus"]
    s = s[s > 1e-3]
    for h in range(REPAIR_H_N1):
        ts.append(pd.DataFrame({"scenario_id": f"n1_{r.outage}", "hour": h, "bus": s.index, "shed_mw": s.values}))
for sid, (shed_t, failed, fam) in REST.items():
    tot = shed_t.sum(axis=1)
    if tot.max() <= 0.1:
        continue
    last = tot.index[tot > 0.1].max()
    st = shed_t.loc[:last]
    st = st.loc[:, (st > 1e-3).any()]
    long = st.reset_index(names="hour").melt(id_vars="hour", var_name="bus", value_name="shed_mw")
    ts.append(long[long.shed_mw > 1e-3].assign(scenario_id=sid))
ts = pd.concat(ts, ignore_index=True)
ts["load_mw"] = ts.bus.map(load_bus)
ts[["scenario_id", "hour", "bus", "load_mw", "shed_mw"]].to_csv(os.path.join(OUT, "bus_shed_timeseries.csv"), index=False)

buses_out = coords.copy()
buses_out["vn_kv"] = net.bus.vn_kv
buses_out["load_mw"] = load_bus
buses_out.index.name = "bus"
buses_out.to_csv(os.path.join(OUT, "network_buses.csv"))
print(f"Exported {meta.shape[0]} scenarios, {len(ts):,} bus-hour rows")

## 7. Notes for the technical note
- **Method.** AC N-1 screening (pandapower) plus a DC-OPF load-shedding LP (SciPy/HiGHS), validated against pandapower's DC power flow.
- **N-k.** A storm-corridor wind field, lognormal tower fragility, Monte Carlo sampling and crew-limited greedy restoration.
- **Limitations to state:**
  - IEEE 118-bus test system, not a real grid.
  - Synthetic thermal ratings (`TARGET_LOADING`, `FLOOR_MVA`).
  - Synthetic geographic placement.
  - The DC approximation ignores reactive power and voltage limits in the load-shed step; AC screening covers those.
  - Corrective re-dispatch is assumed available (unless `REDISPATCH_FRAC` is set).
  - Illustrative fragility and repair parameters.
  - Substations (transformers) are assumed not damaged by wind.
- **Sensitivity worth reporting.** Repeat the analysis with `TARGET_LOADING` = 0.6 and 0.8, `REDISPATCH_FRAC` = 0.2, and `FRAGILITY_MEDIAN` ± 5 m/s.

**Next:** notebook 02 (distribution feeder with rooftop PV).